# 05 - DNSMOS Evaluation (no-reference) of denoising methods

First *quantitative* metric to sit alongside the listening in 03/04. DNSMOS is a
**no-reference** neural MOS — needs only the (denoised) audio, no clean target —
so it's the one intrinsic metric we can compute in Phase 2 (PESQ/STOI/SI-SDR need
clean references that only exist once Phase-4 synthetic pairs are built).

**Methods compared, on the SAME files** (`data/eval_set_big/`, ~25 per noise
category — 144 files; categories with fewer take all available), all run through
the CLI `inference.py`:
- `evalbig_identity`  — raw passthrough (the baseline to beat)
- `evalbig_dsp`       — notch + Butterworth bandpass
- `evalbig_demucs_v4` — htdemucs, subtract vocals

Category labels come from `data/eval_set_big/category.csv` (filename → dominant
noise category). (The original n=5 smoke run used `data/eval_set/`; this is the
scaled version.)

**⚠️ Read DNSMOS honestly:** it was trained to score *speech* enhancement at
16 kHz. Our signal is **breathing/auscultation**, not speech. So a high score
means 'sounds like clean speech', which is a **proxy for noise reduction**, NOT a
measure of diagnostic quality. It will track hiss/voice removal usefully, but
cross-check against the ear (energy/score ≠ audibility — the recurring lesson).

DNSMOS returns 4 scores: **p808** (overall MOS, P.808 model), **SIG** (signal),
**BAK** (background-noise quality), **OVRL** (overall). For denoising, **BAK** and
**OVRL** are the most relevant — removing noise should raise BAK most.

NOTE on SIG for talk files: DNSMOS treats *voice as signal* (it's a speech
metric). So when demucs removes voice, SIG may DROP — that's correct denoising for
us, not damage. Lead with BAK/OVRL; discount SIG on talk/crying categories.

In [ ]:
import sys
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import torch
from torchmetrics.audio.dnsmos import DeepNoiseSuppressionMeanOpinionScore

REPO_ROOT = Path('..').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
SAVED = REPO_ROOT / 'data' / 'saved'

DNSMOS_FS = 16000  # DNSMOS operates at 16 kHz
dnsmos = DeepNoiseSuppressionMeanOpinionScore(fs=DNSMOS_FS, personalized=False)
SCORE_NAMES = ['p808', 'SIG', 'BAK', 'OVRL']

# methods to compare: label -> saved dir (all on data/eval_set_big, same files)
METHODS = {
    'raw': 'evalbig_identity',
    'dsp': 'evalbig_dsp',                       # notch + bandpass
    'dsp_wiener': 'evalbig_dsp_wiener',         # notch + bandpass + wiener
    'demucs_v4': 'evalbig_demucs_v4',
    'combined': 'evalbig_combined',             # demucs -> notch -> bandpass
    'combined_wiener': 'evalbig_combined_wiener',  # demucs -> notch -> bandpass -> wiener
}

# category labels: filename -> dominant noise category, from the eval set CSV
_cat = pd.read_csv(REPO_ROOT / 'data' / 'eval_set_big' / 'category.csv')
CATEGORY = dict(zip(_cat['file'], _cat['category']))
print(f'{len(CATEGORY)} files; categories:',
      _cat['category'].value_counts().to_dict())

In [ ]:
def score_wav(path):
    """Load a wav, resample to 16k (librosa), return DNSMOS 4-vector as dict."""
    y, sr = librosa.load(str(path), sr=None, mono=True)  # native sr
    if sr != DNSMOS_FS:
        y = librosa.resample(y, orig_sr=sr, target_sr=DNSMOS_FS)
    out = dnsmos(torch.from_numpy(y.astype('float32')))
    return dict(zip(SCORE_NAMES, [float(v) for v in out]))


rows = []
for label, sub in METHODS.items():
    d = SAVED / sub / 'test'
    for wav in sorted(d.glob('*.wav')):
        s = score_wav(wav)
        s.update(method=label, file=wav.name, category=CATEGORY.get(wav.name, '?'))
        rows.append(s)

df = pd.DataFrame(rows)[['file', 'category', 'method'] + SCORE_NAMES]
print(f'{len(df)} rows ({df.file.nunique()} files x {df.method.nunique()} methods)')
df.sort_values(['category', 'file', 'method'])

## Summary: mean score per method

Higher = better. Watch **OVRL** and **BAK** (background quality). A good denoiser
should raise these over `raw` without tanking **SIG** (which would mean it ate the
signal).

In [ ]:
summary = df.groupby('method')[SCORE_NAMES].mean().round(3)
# order rows raw -> dsp -> demucs_v4
summary = summary.reindex([m for m in METHODS if m in summary.index])
summary

## Delta vs raw (per metric)

Each method minus the `raw` baseline. Positive = improvement over doing nothing.

In [ ]:
base = summary.loc['raw']
delta = (summary - base).round(3)
delta

## Per-category OVRL (does the win depend on noise type?)

Pivot OVRL by category x method. Expectation from listening: DSP helps the narrow
targets (electric/heart); demucs_v4 helps voice (talk); both should leave
clean_pathology ~unchanged (not harm it).

In [ ]:
for metric in ['OVRL', 'BAK', 'SIG']:
    piv = df.pivot_table(index='category', columns='method', values=metric)
    piv = piv[[m for m in METHODS if m in piv.columns]]
    print(f'--- {metric} by category x method ---')
    display(piv.round(3))

## Notes / takeaways

_Fill in after running (n=144, ~25/category):_
- BAK/OVRL: does demucs_v4 > dsp > raw hold at scale (the n=5 trend)?
- Per-category: DSP best on electric/heart? demucs best on talk/crying/movement?
- clean_pathology: do BOTH leave it ~unchanged (not harm the already-clean file)?
- SIG on talk/crying: expected to drop for demucs (voice=signal to DNSMOS) — that's
  correct denoising for us, NOT damage. Confirm BAK rises there.
- Where DNSMOS disagrees with the ear (03/04): trust the ear, note the disagreement —
  it's a speech metric on breathing; those disagreements are reportable.
- This is the no-reference half. Full-reference (PESQ/STOI/SI-SDR) + classifier-gain
  come later (Phase-4 pairs / separate dnd eval).